In [ ]:
# ---- EXPERIMENT CONFIG: edit this cell for each run --------------------------
# SYMSOL I with 10 % of the training data (Howell et al. 2023, Table 2; 4,500 of 45,000 views per shape: tet, cube, icosa, cone, cyl; one model for
# all five), Wigner-D harmonics regression (Lee & Cho 2024, arXiv 2411.00543, --model wignerd) on the Image2Sphere protocol: ResNet-50 (ImageNet),
# SGD + Nesterov 0.9, batch 64, 40 epochs, lr x0.1 every 15 epochs, no EMA, no augmentation. A fresh equivalent pose is the regression target at
# every read, so the network learns the average of D(R) over the equivalents, i.e. a band-limited (lmax 6) density over SO(3).
# Metrics: nearest-equivalent median error per epoch (first 1,000 test views per shape); at the end the SYMSOL log-likelihood on the full test set
# (5,000 views per shape, 8 equivalents each), Image2Sphere's grid protocol (P(Voronoi cell of the label on the level-5 HEALPix grid) * N / pi^2, capped
# at log(N/pi^2) - log(#equivalents)): at softmax temperature 1, and at the temperature calibrated on each shape's first 500 test views (reported on the rest).
# Howell et al. Table 2, 10 % column: IPDF -7.94, I2S 2.98, Howell 3.01. The paper (full data, Table 6): Wigner loss 2.54, I2S loss 3.41.
# Attach: syfry5suvzovvakmuj/symsol1-10pct-fulltest, ga-research-offline-deps and ONE repo snapshot.
COMMON = ['--model', 'wignerd', '--encoder', 'resnet50', '--dataset', 'symsol', '--symsol_set', '1', '--symsol_train_views', '4500',
          '--symsol_test_views', '1000', '--symsol_ll_views', '5000', '--no-use_warp', '--no-use_synth', '--no-ema', '--batch_size', '64',
          '--optimizer', 'sgd', '--lr_scaling', 'none', '--lr_schedule', 'step', '--lr_step_epochs', '15', '--n_epochs', '40',
          '--grad_clip', '0', '--i2s_eval_rec_level', '5', '--num_workers', '8']
VARIANTS = {'wignerd_symsol1_10pct_lr0p1': ['--lr', '0.1'],       # the paper's Pascal3D+ learning rate
            'wignerd_symsol1_10pct_lr0p01': ['--lr', '0.01']}
SNAPSHOT_MARKER = 'symsol_ll_views'   # the snapshot's config.py must contain it (stale-snapshot guard)
ARGS = COMMON   # the dependency cell checks it for --vector_field gatr


In [ ]:
import glob, os, shutil, subprocess, tarfile

# The snapshot dataset is either auto-extracted by Kaggle (a GA-Research/ tree) or still a tar.gz.
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
markers = glob.glob('/kaggle/input/**/pose3d/pose3d/config.py', recursive=True)
if markers:
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    repo_root = markers[0][:markers[0].index('/pose3d/pose3d/config.py')]
    shutil.copytree(repo_root, '/kaggle/working/GA-Research')
else:
    tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
    assert len(tgzs) == 1, f'attach exactly one repo snapshot, found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
print(sorted(os.listdir('/kaggle/working/GA-Research')))
config_src = open('/kaggle/working/GA-Research/pose3d/pose3d/config.py').read()
assert SNAPSHOT_MARKER in config_src, f'stale snapshot: {SNAPSHOT_MARKER!r} is not in config.py'

In [ ]:
# Libraries Kaggle lacks, from the offline wheels (no index, no deps: the image's torch stays).
deps = next(p for p in glob.glob('/kaggle/input/**/ga-research-offline-deps', recursive=True)
            + glob.glob('/kaggle/input/ga-research-offline-deps'))
wheels = sorted(glob.glob(os.path.join(deps, '*.whl')))
# GATr (the reference vector field) and einops: from any mounted *ga-research-gatr-deps* copy.
gatr_deps = glob.glob('/kaggle/input/**/ga-research-gatr-deps', recursive=True)
wheels += sorted(w for d in gatr_deps[:1] for w in glob.glob(os.path.join(d, '*.whl')))
print('wheels:', [os.path.basename(w) for w in wheels])
subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)

# torchvision's resnet101 DEFAULT weights, so the pretrained backbone needs no download.
ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)
shutil.copy(os.path.join(deps, 'resnet101-cd907fc2.pth'), os.path.join(ckpt_dir, 'resnet101-cd907fc2.pth'))
for w in glob.glob('/kaggle/input/**/resnet50-11ad3fa6.pth', recursive=True)[:1]:   # ImageNet weights of --encoder resnet50
    shutil.copy(w, os.path.join(ckpt_dir, 'resnet50-11ad3fa6.pth'))

import importlib, sys
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford', 'opt_einsum']:
    importlib.import_module(mod)
if '--vector_field' in ARGS and ARGS[ARGS.index('--vector_field') + 1] == 'gatr':
    sys.path.insert(0, '/kaggle/working/GA-Research/pose3d')
    from pose3d.models.gatr_denoiser import _import_gatr
    print('GATr:', _import_gatr())   # fails here, not hours in, when the gatr-deps copy is missing
print('python', sys.version.split()[0], '| all offline deps import cleanly')

In [ ]:
# No internet: wandb writes local files only; the trainer also prints WANDB_SYNC lines per epoch.
os.environ['WANDB_MODE'] = 'offline'

In [ ]:
import sys, os, glob, subprocess
os.chdir('/kaggle/working/GA-Research/pose3d')   # the package is found from here, no install needed
found = glob.glob('/kaggle/input/**/symsol/test/rotations.npz', recursive=True)
DATA = found[0][:-len('/symsol/test/rotations.npz')]
print('SYMSOL root:', DATA, flush=True)
failed = []
for name, extra in VARIANTS.items():
    cmd = [sys.executable, '-u', '-m', 'pose3d', f'--path_to_datasets={DATA}', '--platform=kaggle', f'--run_name={name}', *COMMON, *extra]
    print(' '.join(cmd), flush=True)
    if subprocess.run(cmd, env={**os.environ}).returncode:
        failed.append(name)
assert not failed, f'failed variants: {failed}'   # a crash marks the Kaggle run as failed instead of passing silently
